# Allocation & Concentration

Where the money *is* — sliced from broker down to individual lot — and how concentrated
that makes you. A handful of names doing most of the work is the default state of a real
portfolio; these views make the tilt impossible to miss.

*Click a sunburst/treemap wedge to zoom in; click the center to zoom back out.*

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} across {positions['account_name'].nunique()} accounts "
      f"· {len(history)} trading days {history.index.min():%Y-%m-%d}→{history.index.max():%Y-%m-%d}")

## Sunburst — broker → account → asset class → holding

The whole portfolio in one ring. Each level subdivides the one inside it; arc length is
dollar value.

In [ ]:
p = positions.copy()
p["leaf"] = p["symbol"].where(~p["is_cash"], "Cash")
fig = px.sunburst(p, path=["broker", "account_name", "asset_class", "leaf"],
                  values="market_value", color="asset_class",
                  color_discrete_sequence=PALETTE,
                  title="Portfolio allocation — click to drill in")
fig.update_traces(insidetextorientation="radial",
                  hovertemplate="<b>%{label}</b><br>$%{value:,.0f}<br>%{percentRoot:.1%} of portfolio<extra></extra>")
fig.update_layout(height=620)
fig.show()

## Treemap — same hierarchy, packed by size

Easier to compare areas than arcs. Hover for dollar value and share of the portfolio.

In [ ]:
fig = px.treemap(p, path=[px.Constant("Portfolio"), "asset_class", "leaf"],
                 values="market_value", color="asset_class",
                 color_discrete_sequence=PALETTE, title="Allocation treemap")
fig.update_traces(textinfo="label+percent root",
                  hovertemplate="<b>%{label}</b><br>$%{value:,.0f}<br>%{percentRoot:.1%}<extra></extra>")
fig.update_layout(height=560)
fig.show()

## Concentration gauges

**Effective number of holdings** = 1 / HHI — how many *equally sized* positions would give
the same concentration (you have far fewer effective bets than line items). Plus the share
of the portfolio in your single largest and top-five names.

In [ ]:
w = A.portfolio_weights(positions, include_cash=True)
c = A.concentration_metrics(w)
fig = make_subplots(rows=1, cols=3, specs=[[{"type": "indicator"}]*3])
fig.add_trace(go.Indicator(mode="gauge+number", value=c["effective_holdings"],
    title={"text": "Effective holdings"}, gauge={"axis": {"range": [0, len(w)]},
    "bar": {"color": PALETTE[0]}}), row=1, col=1)
fig.add_trace(go.Indicator(mode="gauge+number", value=c["top1_weight"] * 100,
    title={"text": "Top-1 weight %"}, number={"suffix": "%"},
    gauge={"axis": {"range": [0, 100]}, "bar": {"color": PALETTE[1]}}), row=1, col=2)
fig.add_trace(go.Indicator(mode="gauge+number", value=c["top5_weight"] * 100,
    title={"text": "Top-5 weight %"}, number={"suffix": "%"},
    gauge={"axis": {"range": [0, 100]}, "bar": {"color": PALETTE[2]}}), row=1, col=3)
fig.update_layout(height=300, title=f"Concentration · HHI = {c['hhi']:.3f} "
                  f"(1.0 = a single holding)")
fig.show()

## Allocation by account, broken out by asset class

Which account holds what. Tax-advantaged accounts (IRA/HSA/401k) vs taxable changes what
you'd *do* about any given tilt.

In [ ]:
by = (positions.groupby(["account_name", "asset_class"])["market_value"].sum()
      .reset_index())
order = positions.groupby("account_name")["market_value"].sum().sort_values().index
fig = px.bar(by, y="account_name", x="market_value", color="asset_class", orientation="h",
             color_discrete_sequence=PALETTE, category_orders={"account_name": list(order)},
             title="Account composition", hover_data={"market_value": ":$,.0f"})
fig.update_layout(height=460, xaxis_tickprefix="$", barmode="stack",
                  yaxis_title="", xaxis_title="market value")
fig.show()

## Concentration drift — has it gotten worse?

Because positions are a fold over the dated ledger, we can re-derive your holdings at each
past quarter-end, price them with that day's prices, and watch concentration evolve. Rising
top-1 weight / falling effective-N means you're getting *less* diversified over time.

In [ ]:
entries, _, _ = ledger.load()
tmap = {r["symbol"]: A._price_ticker(r, history)
        for _, r in positions[~positions["is_cash"]].iterrows()}
tmap = {s: t for s, t in tmap.items() if t}

rows = []
for d in pd.date_range(history.index.min(), history.index.max(), freq="QE"):
    pit = ledger.positions_dataframe(entries, date=d)
    shares = pit[~pit["is_cash"]].groupby("symbol")["quantity"].sum()
    px_d = history.loc[:d].ffill().iloc[-1]
    vals = pd.Series({s: q * px_d.get(tmap[s], np.nan) for s, q in shares.items() if s in tmap}).dropna()
    if vals.sum() > 0:
        m = A.concentration_metrics(vals)
        rows.append({"date": d, "effective_holdings": m["effective_holdings"],
                     "top1_weight": m["top1_weight"]})
drift = pd.DataFrame(rows).set_index("date")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=drift.index, y=drift["effective_holdings"], name="effective holdings",
                line=dict(color=PALETTE[0], width=3))
fig.add_scatter(x=drift.index, y=drift["top1_weight"], name="top-1 weight",
                line=dict(color=PALETTE[1], width=3, dash="dot"), secondary_y=True)
fig.update_yaxes(title_text="effective holdings", secondary_y=False)
fig.update_yaxes(title_text="top-1 weight", tickformat=".0%", secondary_y=True)
fig.update_layout(height=420, title="Concentration over time (priced sleeve)", hovermode="x unified")
fig.show()

---
*The concentration-drift chart covers only holdings with a usable price series; cash and a
few un-priceable lots are excluded from those weights (the sunburst/treemap include
everything).*